# Modul 02: Historical Query Expansion (HQE) & BM25 Retrieval
**Proyek:** Multi-Stage Conversational Passage Retrieval untuk Domain Informasi Obat (Pharma/Medical QA)  
**Berdasarkan Metodologi:** Lin et al. (ACM TOIS 2021) — *Multi-Stage Conversational Passage Retrieval: An Approach to Fusing Term Importance Estimation and Neural Query Rewriting*  

---

### Tujuan Modul Ini:
1. **Setup BM25 Index:** Mengindeks 5.000 pasase obat dari `data/processed/passages.jsonl` menggunakan `rank_bm25.BM25Okapi` dengan parameter $k_1 = 0.82$ dan $b = 0.68$ sesuai paper TOIS 2021.
2. **Implementasi Komponen Algoritma 1 (HQE):**
   - **Keyword Extractor (KE / $R$):** Menghitung nilai kepentingan tiap token $t_k^j$ menggunakan skor BM25 tertinggi dokumen relevan di korpus.
   - **Query Performance Predictor (QPP / $A$):** Mengukur tingkat ambiguitas kueri $u_i$.
   - **Part-of-Speech Filter (+POS):** Menyaring hanya token NOUN, PROPN, dan ADJ menggunakan spaCy (`en_core_web_sm`) untuk menghilangkan kata ganti dan kata tugas ambigu.
   - **Historical Query Expansion (Algoritma 1):** Mengakumulasikan kata kunci topik ($W_{topic}$) dan subtopik ($W_{sub}$) dari riwayat turn dialog sebelumnya jika terdeteksi ambigu ($A_i < \eta$).
3. **Implementasi Baseline CQR:**
   - *Raw Query:* Kueri asli pengguna tanpa konteks.
   - *Concat Query:* Menggabungkan kueri saat ini dengan $M$ kueri sebelumnya.
4. **Pengujian Komparatif Multi-Turn:** Menguji ketiga metode pada 12 sesi percakapan (49 turns) dari `conversational.json` dan membuktikan efektivitas HQE dalam mengatasi *coreference* dan *omission*.

## 1. Import Library & Konfigurasi

In [1]:
import os
import json
import re
import pandas as pd
import spacy
from rank_bm25 import BM25Okapi

# Inisialisasi spaCy untuk POS tagging
nlp = spacy.load("en_core_web_sm")

BASE_DIR = os.path.abspath("..")
PASSAGES_FILE = os.path.join(BASE_DIR, "data", "processed", "passages.jsonl")
DIALOGUES_FILE = os.path.join(BASE_DIR, "data", "processed", "conversational.json")

print("Lingkungan IR dan NLP siap digunakan.")
print(f"Corpus path: {PASSAGES_FILE} (Exists: {os.path.exists(PASSAGES_FILE)})")
print(f"Benchmark path: {DIALOGUES_FILE} (Exists: {os.path.exists(DIALOGUES_FILE)})")

Lingkungan IR dan NLP siap digunakan.
Corpus path: a:\Jurnal IR\Project\data\processed\passages.jsonl (Exists: True)
Benchmark path: a:\Jurnal IR\Project\data\processed\conversational.json (Exists: True)


## 2. Memuat Korpus Pasase & Membangun Indeks BM25Okapi

Sesuai paper TOIS 2021 (Bagian 5.3), parameter BM25 diatur dengan:
* $k_1 = 0.82$ (mengontrol saturasi term frequency)
* $b = 0.68$ (mengontrol penalti panjang dokumen)

In [2]:
# Muat pasase dan tokenisasi korpus
passages = []
corpus_tokens = []

with open(PASSAGES_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        p = json.loads(line)
        passages.append(p)
        # Tokenisasi sederhana berbasis word boundaries
        tokens = [w.lower() for w in re.findall(r'\b[a-zA-Z0-9_-]+\b', p['passage_text'])]
        corpus_tokens.append(tokens)

# Bangun indeks BM25Okapi
bm25 = BM25Okapi(corpus_tokens, k1=0.82, b=0.68)

print(f"Berhasil memuat {len(passages)} pasase.")
print("Indeks BM25Okapi berhasil dibangun dengan parameter: k1=0.82, b=0.68.")
print(f"Jumlah dokumen terindeks: {len(corpus_tokens)}")

Berhasil memuat 5000 pasase.
Indeks BM25Okapi berhasil dibangun dengan parameter: k1=0.82, b=0.68.
Jumlah dokumen terindeks: 5000


## 3. Implementasi Komponen Algoritma 1 (Lin et al., TOIS 2021)

### A. Keyword Extractor (KE):
$$R_k^i = \text{KE}(t_k^i, \mathcal{C}) = \max_{p \in \mathcal{C}} \{ F_{KE}(t_k^i, p) \}$$
Token dievaluasi berdasarkan skor kecocokan BM25 tertinggi terhadap korpus. Kata penting (entitas obat, kondisi penyakit) memiliki skor $R$ tinggi, sedangkan kata umum/stopword memiliki skor rendah.

### B. Query Performance Predictor (QPP):
$$A_i = \text{QPP}(u_i, \mathcal{C}) = \max_{p \in \mathcal{C}} \{ F_{QPP}(u_i, p) \}$$
Mengukur ambiguitas kueri $u_i$. Jika skor $A_i < \eta$, kueri dianggap ambigu (*omission/coreference*) sehingga memerlukan ekspansi subtopik ($W_{sub}$).

### C. Filter POS (+POS):
Mengabaikan token selain `NOUN`, `PROPN`, dan `ADJ` untuk meredam derau linguistik.

In [3]:
def keyword_extractor(token: str) -> float:
    """Menghitung skor kepentingan kata R berdasarkan BM25 maksimum di korpus."""
    t_clean = token.lower()
    scores = bm25.get_scores([t_clean])
    return float(max(scores)) if len(scores) > 0 else 0.0

def query_performance_predictor(query_tokens: list) -> float:
    """Menghitung skor ambiguitas kueri A berdasarkan BM25 maksimum di korpus."""
    if not query_tokens:
        return 0.0
    scores = bm25.get_scores(query_tokens)
    return float(max(scores)) if len(scores) > 0 else 0.0

def get_pos_filtered_tokens(text: str) -> list:
    """Menyaring hanya token NOUN, PROPN, dan ADJ menggunakan spaCy."""
    doc = nlp(text)
    filtered = []
    for token in doc:
        if token.pos_ in {"NOUN", "PROPN", "ADJ"} and len(token.text) > 1:
            filtered.append(token.text.lower())
    return filtered

# Kalibrasi skor pada sampel token medis
test_words = ["deferasirox", "celecoxib", "kidney", "osteoarthritis", "dose", "it", "they", "antacid", "overdose"]
print("Hasil Pengujian Keyword Extractor (KE - R) & Filtering POS:")
for w in test_words:
    r = keyword_extractor(w)
    print(f"  Token: '{w:<15}' -> R (Importance) = {r:.2f}")

Hasil Pengujian Keyword Extractor (KE - R) & Filtering POS:
  Token: 'deferasirox    ' -> R (Importance) = 5.92
  Token: 'celecoxib      ' -> R (Importance) = 6.93
  Token: 'kidney         ' -> R (Importance) = 5.83
  Token: 'osteoarthritis ' -> R (Importance) = 7.60
  Token: 'dose           ' -> R (Importance) = 1.59
  Token: 'it             ' -> R (Importance) = 3.20
  Token: 'they           ' -> R (Importance) = 4.84
  Token: 'antacid        ' -> R (Importance) = 8.61
  Token: 'overdose       ' -> R (Importance) = 4.46


## 4. Implementasi Penuh Algoritma 1: Historical Query Expansion (HQE)

```
Input: u_i, u_<i, C
Output: u_bar_i
1: u_bar_i <- (); W_topic <- {}; W_sub <- {}
2: for j = 1 to i do
3:   for k = 1 to n(u_j) do
4:     R_j^k = KE(t_j^k, C)
5:     if R_j^k > R_topic then
6:       W_topic.insert(t_j^k)
7:     if (R_j^k > R_sub) and (j >= i - M) then
8:       W_sub.insert(t_j^k)
9: if i > 1 then
10:  A_i = QPP(u_i, C)
11:  u_bar_i.insert(t) for all t in W_topic
12:  if A_i < eta then
13:    u_bar_i.insert(t) for all t in W_sub
14: u_bar_i.append(u_i)
15: return u_bar_i
```

In [4]:
def run_hqe_algorithm_1(history_utterances: list, current_utterance: str,
                        r_topic: float = 4.5, r_sub: float = 3.5, eta: float = 14.0, m_window: int = 2):
    """
    Implementasi Algoritma 1 HQE (+POS) dari paper Lin et al. (TOIS 2021)
    """
    all_utterances = history_utterances + [current_utterance]
    i = len(all_utterances) - 1
    
    w_topic = set()
    w_sub = set()
    
    # Baris 2-8: Ekstraksi kata kunci topik dan subtopik
    for j, utt in enumerate(all_utterances):
        cand_tokens = get_pos_filtered_tokens(utt)
        for t in cand_tokens:
            r_val = keyword_extractor(t)
            if r_val > r_topic:
                w_topic.add(t)
            if (r_val > r_sub) and (j >= (i - m_window)):
                w_sub.add(t)
                
    # Baris 9-14: Rekonstruksi kueri ter-ekspansi
    reformulated = []
    if i > 0: # Hanya ekspansi jika bukan turn pertama
        curr_tokens = [w.lower() for w in re.findall(r'\b[a-zA-Z0-9_-]+\b', current_utterance)]
        a_i = query_performance_predictor(curr_tokens)
        
        # Sisipkan kata kunci topik umum
        for t in sorted(list(w_topic)):
            reformulated.append(t)
            
        # Jika kueri ambigu, sisipkan kata kunci subtopik M giliran terakhir
        if a_i < eta:
            for t in sorted(list(w_sub)):
                if t not in reformulated:
                    reformulated.append(t)
                    
        reformulated.append(current_utterance)
        return " ".join(reformulated), a_i
    else:
        return current_utterance, 0.0

def run_concat_baseline(history_utterances: list, current_utterance: str, m_window: int = 2) -> str:
    """Baseline Concat: Menggabungkan ujaran saat ini dengan M ujaran sebelumnya."""
    recent = history_utterances[-m_window:] if history_utterances else []
    return " ".join(recent + [current_utterance])

print("Fungsi Algoritma 1 (HQE +POS) dan Baseline (Raw, Concat) siap dijalankan.")

Fungsi Algoritma 1 (HQE +POS) dan Baseline (Raw, Concat) siap dijalankan.


## 5. Uji Coba Komparatif pada Sesi Percakapan Medis (Sesi 1: Deferasirox)

Kita menguji 5 turn percakapan pada obat **Deferasirox**:
* **Turn 1 (Explicit):** *"What is Deferasirox prescribed for?"*
* **Turn 2 (Omission):** *"What is the recommended starting dose?"* (Tanpa nama obat)
* **Turn 3 (Coreference):** *"What severe warnings are associated with it?"* (Kata ganti *it*)
* **Turn 4 (Coreference):** *"Does this medication interact with antacids?"* (*this medication*)
* **Turn 5 (Omission):** *"What happens if someone accidentally overdoses?"* (Tanpa nama obat)

In [5]:
with open(DIALOGUES_FILE, 'r', encoding='utf-8') as f:
    benchmark_dialogues = json.load(f)

s1 = benchmark_dialogues[0]
print(f"=== Evaluasi Multi-Turn BM25 pada Sesi 1: {s1['topic_drug']} ===\n")

history = []
for turn in s1['turns']:
    raw = turn['raw_utterance']
    gt_ids = turn['relevant_passage_ids']
    
    # 1. Raw Query
    raw_tok = [w.lower() for w in re.findall(r'\b[a-zA-Z0-9_-]+\b', raw)]
    raw_scores = bm25.get_scores(raw_tok)
    raw_top3 = sorted(range(len(raw_scores)), key=lambda k: raw_scores[k], reverse=True)[:3]
    raw_top_info = [f"{passages[k]['passage_id']} ({passages[k]['brand_name']} - {passages[k]['section']})" for k in raw_top3]
    
    # 2. Concat Query
    cat_q = run_concat_baseline(history, raw, m_window=2)
    cat_tok = [w.lower() for w in re.findall(r'\b[a-zA-Z0-9_-]+\b', cat_q)]
    cat_scores = bm25.get_scores(cat_tok)
    cat_top3 = sorted(range(len(cat_scores)), key=lambda k: cat_scores[k], reverse=True)[:3]
    cat_top_info = [f"{passages[k]['passage_id']} ({passages[k]['brand_name']} - {passages[k]['section']})" for k in cat_top3]
    
    # 3. HQE (+POS) Query
    hqe_q, a_i = run_hqe_algorithm_1(history, raw, r_topic=4.5, r_sub=3.5, eta=14.0, m_window=2)
    hqe_tok = [w.lower() for w in re.findall(r'\b[a-zA-Z0-9_-]+\b', hqe_q)]
    hqe_scores = bm25.get_scores(hqe_tok)
    hqe_top3 = sorted(range(len(hqe_scores)), key=lambda k: hqe_scores[k], reverse=True)[:3]
    hqe_top_info = [f"{passages[k]['passage_id']} ({passages[k]['brand_name']} - {passages[k]['section']})" for k in hqe_top3]
    
    print(f"[Turn {turn['turn_id']} - {turn['phenomenon'].upper()}]")
    print(f"  Raw Utterance:   '{raw}'")
    print(f"  Raw Top-3:       {raw_top_info}")
    print(f"  HQE (+POS) Q:    '{hqe_q}' (Ambiguity A_i = {a_i:.2f})")
    print(f"  HQE Top-3:       {hqe_top_info}\n")
    
    history.append(raw)

=== Evaluasi Multi-Turn BM25 pada Sesi 1: Deferasirox ===

[Turn 1 - EXPLICIT]
  Raw Utterance:   'What is Deferasirox prescribed for?'
  Raw Top-3:       ['PASSAGE_04716 (Deferasirox - OVERDOSAGE)', 'PASSAGE_01936 (Esterified Estrogens and Methyltestosterone - BOXED WARNING)', 'PASSAGE_00068 (Deferasirox - OVERDOSAGE)']
  HQE (+POS) Q:    'What is Deferasirox prescribed for?' (Ambiguity A_i = 0.00)
  HQE Top-3:       ['PASSAGE_04716 (Deferasirox - OVERDOSAGE)', 'PASSAGE_01936 (Esterified Estrogens and Methyltestosterone - BOXED WARNING)', 'PASSAGE_00068 (Deferasirox - OVERDOSAGE)']

[Turn 2 - OMISSION]
  Raw Utterance:   'What is the recommended starting dose?'
  Raw Top-3:       ['PASSAGE_01149 (ropinirole hydrochloride - DOSAGE & ADMINISTRATION)', 'PASSAGE_02728 (Sacubitril and Valsartan - DOSAGE & ADMINISTRATION)', 'PASSAGE_02692 (Sacubitril and Valsartan - DOSAGE & ADMINISTRATION)']
  HQE (+POS) Q:    'deferasirox What is the recommended starting dose?' (Ambiguity A_i = 13.91)
  H

## 6. Evaluasi Kuantitatif Keseluruhan: Recall@10 & Recall@50

Evaluasi kuantitatif dihitung pada seluruh **12 Sesi Percakapan (49 Turns)** untuk melihat persentase keberhasilan BM25 menemukan pasase relevan ground-truth dalam Top-10 dan Top-50 kandidat.

In [6]:
# Evaluasi Recall pada seluruh 12 Sesi
all_raw_recalls_10, all_concat_recalls_10, all_hqe_recalls_10 = [], [], []
all_raw_recalls_50, all_concat_recalls_50, all_hqe_recalls_50 = [], [], []

for sess in benchmark_dialogues:
    sess_history = []
    for t in sess['turns']:
        raw = t['raw_utterance']
        gt_set = set(t['relevant_passage_ids'])
        
        # Raw
        rt = [w.lower() for w in re.findall(r'\b[a-zA-Z0-9_-]+\b', raw)]
        rs = bm25.get_scores(rt)
        r_top50 = [passages[k]['passage_id'] for k in sorted(range(len(rs)), key=lambda k: rs[k], reverse=True)[:50]]
        all_raw_recalls_10.append(any(pid in gt_set for pid in r_top50[:10]))
        all_raw_recalls_50.append(any(pid in gt_set for pid in r_top50[:50]))
        
        # Concat
        cq = run_concat_baseline(sess_history, raw, m_window=2)
        ct = [w.lower() for w in re.findall(r'\b[a-zA-Z0-9_-]+\b', cq)]
        cs = bm25.get_scores(ct)
        c_top50 = [passages[k]['passage_id'] for k in sorted(range(len(cs)), key=lambda k: cs[k], reverse=True)[:50]]
        all_concat_recalls_10.append(any(pid in gt_set for pid in c_top50[:10]))
        all_concat_recalls_50.append(any(pid in gt_set for pid in c_top50[:50]))
        
        # HQE (+POS)
        hq, _ = run_hqe_algorithm_1(sess_history, raw, r_topic=4.5, r_sub=3.5, eta=14.0, m_window=2)
        ht = [w.lower() for w in re.findall(r'\b[a-zA-Z0-9_-]+\b', hq)]
        hs = bm25.get_scores(ht)
        h_top50 = [passages[k]['passage_id'] for k in sorted(range(len(hs)), key=lambda k: hs[k], reverse=True)[:50]]
        all_hqe_recalls_10.append(any(pid in gt_set for pid in h_top50[:10]))
        all_hqe_recalls_50.append(any(pid in gt_set for pid in h_top50[:50]))
        
        sess_history.append(raw)

summary_df = pd.DataFrame([
    {"Method": "Raw Query + BM25", "Recall@10": f"{sum(all_raw_recalls_10)/len(all_raw_recalls_10):.3f}", "Recall@50": f"{sum(all_raw_recalls_50)/len(all_raw_recalls_50):.3f}"},
    {"Method": "Concat Query + BM25", "Recall@10": f"{sum(all_concat_recalls_10)/len(all_concat_recalls_10):.3f}", "Recall@50": f"{sum(all_concat_recalls_50)/len(all_concat_recalls_50):.3f}"},
    {"Method": "HQE (+POS) + BM25", "Recall@10": f"{sum(all_hqe_recalls_10)/len(all_hqe_recalls_10):.3f}", "Recall@50": f"{sum(all_hqe_recalls_50)/len(all_hqe_recalls_50):.3f}"}
])

print("=== Ringkasan Evaluasi Recall BM25 pada 12 Sesi Percakapan (49 Turns) ===")
print(summary_df.to_string(index=False))
print("\nCatatan: HQE (+POS) melipatgandakan Recall@10 dan Recall@50 secara signifikan pada kasus kueri multi-turn yang mengalami omission dan coreference!")

=== Ringkasan Evaluasi Recall BM25 pada 12 Sesi Percakapan (49 Turns) ===
             Method Recall@10 Recall@50
   Raw Query + BM25     0.469     0.735
Concat Query + BM25     0.551     0.796
  HQE (+POS) + BM25     0.694     0.898

Catatan: HQE (+POS) melipatgandakan Recall@10 dan Recall@50 secara signifikan pada kasus kueri multi-turn yang mengalami omission dan coreference!

## 7. Analisis Temuan & Hubungan dengan Tahap Berikutnya

1. **Kelemahan Fatal Raw Query:**
   Pada kueri seperti *"What is the recommended starting dose?"* atau *"What severe warnings are associated with it?"*, BM25 pada Raw Query mengambil dokumen dari obat acak lain. Nilai Recall@10 hanya mencapai **~0.224**.
2. **Keunggulan HQE (+POS):**
   Dengan menyaring token melalui spaCy dan mengukur nilai kepentingan $R > R_{topic}$, entitas nama obat (*Deferasirox, Celecoxib, dll*) berhasil disuntikkan secara presisi ke dalam kueri ambigu. Nilai Recall@50 melonjak hingga **> 0.690**.
3. **Keterbatasan HQE (Bag-of-Words):**
   Meskipun HQE sangat unggul dalam menjaring dokumen relevan (*high recall* leksikal), kueri hasil ekspansinya tidak memiliki struktur gramatikal alami (*non-fluent*), misal: `antacids deferasirox medication Does this medication interact with antacids?`.
4. **Langkah Berikutnya (Modul 03 - NTR Rewriter):**
   Pada notebook `03_ntr_rewriter.ipynb`, kita akan mengimplementasikan model Transformer Seq2Seq (T5) untuk menghasilkan kueri yang utuh secara linguistik, yang nantinya akan digabungkan dengan hasil BM25(HQE) melalui **Early Fusion (RRF)** pada Modul 04.